# indian_equities_strat_001

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (130).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `indian_equities` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 6 |
| Detected functions | identify_fvg, find_order_blocks, identify_liquidity_sweeps, calculate_fibonacci_levels, filter_session, apply_trading_logic, backtest_strategy |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd
import pandas_datareader as pdr
from datetime import datetime
import matplotlib.pyplot as plt

In [ ]:
# Now you can load the entire rows
data = pd.read_parquet('B&F_OCT_2024.parquet')
data

In [ ]:
import pandas as pd

# Convert Date_Time to datetime
data['Date_Time'] = pd.to_datetime(data['Date_Time'])

# Define filtering criteria
start_date = '2024-10-01 09:15:00'
end_date = '2024-10-31 15:31:00'
tickers_to_include = ['SBIN.NSE']

# Apply filtering
df = data[
    (data['Date_Time'] >= start_date) &
    (data['Date_Time'] <= end_date) &
    (data['Ticker'].isin(tickers_to_include))
]

# Print the filtered dataset with all columns
df


In [ ]:
import pandas as pd
import numpy as np

# Assuming your DataFrame is named 'df' and includes columns: 'LTP', 'BuyPrice', 'BuyQty', 'SellPrice', 'SellQty', 'LTQ'

# Add a datetime index if not already present
df['datetime'] = pd.to_datetime(df.index)  # Replace this with your actual timestamp column if available
df.set_index('datetime', inplace=True)

# Parameters for the strategy
fvg_threshold = 0.002  # Threshold for Fair Value Gap
fib_levels = (0.62, 0.79)  # Optimal Trade Entry levels
session_start = '09:15'
session_end = '15:30'

# Step 1: Identify Fair Value Gaps (using LTP)
def identify_fvg(df, threshold):
    df['fvg'] = np.where(
        (df['LTP'].shift(1) < df['LTP'].shift(-1)) & 
        (np.abs(df['LTP'].shift(-1) - df['LTP']) > threshold), 
        True, False
    )
    return df

# Step 2: Find Order Blocks (using BuyPrice and SellPrice)
def find_order_blocks(df):
    df['order_block'] = np.where(
        (df['LTP'] < df['BuyPrice']) & (df['LTP'] == df['LTP'].rolling(150).min()), 
        'bearish', 
        np.where((df['LTP'] > df['SellPrice']) & (df['LTP'] == df['LTP'].rolling(150).max()), 'bullish', None)
    )
    return df

# Step 3: Identify Liquidity Sweeps (using BuyPrice and SellPrice)
def identify_liquidity_sweeps(df):
    df['liquidity_sweep'] = np.where(
        (df['LTP'] < df['BuyPrice'].rolling(5).min()) | (df['LTP'] > df['SellPrice'].rolling(250).max()), 
        True, False
    )
    return df

# Step 4: Optimal Trade Entry (using Fibonacci levels)
def calculate_fibonacci_levels(df):
    df['fib_high'] = df['LTP'].rolling(500).max()
    df['fib_low'] = df['LTP'].rolling(500).min()
    df['fib_retracement'] = (df['LTP'] - df['fib_low']) / (df['fib_high'] - df['fib_low'])
    df['ote_zone'] = (df['fib_retracement'] >= fib_levels[0]) & (df['fib_retracement'] <= fib_levels[1])
    return df

# Step 5: Power of Three - Session Timing
def filter_session(df, start, end):
    df['in_session'] = df.index.time >= pd.to_datetime(start).time()
    df['in_session'] &= df.index.time <= pd.to_datetime(end).time()
    return df

# Apply all steps to the DataFrame
df = identify_fvg(df, fvg_threshold)
df = find_order_blocks(df)
df = identify_liquidity_sweeps(df)
df = calculate_fibonacci_levels(df)
df = filter_session(df, session_start, session_end)

def apply_trading_logic(df):
    df['trade_signal'] = np.where(
        (df['fvg']) &
        (df['order_block'].notna()) &
        (df['liquidity_sweep']) &
        (df['ote_zone']) &
        (df['in_session']),
        np.where(df['order_block'] == 'bullish', 'Buy', 'Sell'),
        None
    )
    return df

df = apply_trading_logic(df)

# Display or save the signals
trade_signals = df[df['trade_signal'].notna()]
print(trade_signals[['LTP', 'BuyPrice', 'SellPrice', 'trade_signal']])

# Save the trade signals to a CSV
trade_signals.to_csv("trade_signals.csv", index=True)


In [ ]:
def backtest_strategy(df, initial_capital=100000, brokerage_rate=0.0001, session_close="15:30:00"):
    """
    Backtests the strategy based on the generated trade signals and closes all trades by session end.

    Parameters:
    - df: DataFrame containing the trading data with 'trade_signal' column.
    - initial_capital: Starting capital for the backtest.
    - brokerage_rate: Brokerage fee as a fraction (e.g., 0.0001 for 0.01%).
    - session_close: Time to forcefully close all open positions (default is 15:30:00).

    Returns:
    - tradebook: DataFrame containing detailed trade information.
    - final_portfolio_value: Total portfolio value after all trades.
    """
    # Initialize variables
    capital = initial_capital
    tradebook = []
    position = None  # Track the current position ('Buy' or 'Sell')
    entry_price = None
    qty = None

    # Iterate through the DataFrame
    for i, row in df.iterrows():
        signal = row['trade_signal']
        ltp = row['LTP']
        current_time = i.strftime('%H:%M:%S')

        # Open a Buy position if signal is 'Buy' and no position is open
        if signal == 'Buy' and position is None:
            entry_price = ltp
            position = 'Buy'
            qty = capital / ltp  # Buy as many units as possible with current capital
            tradebook.append({
                'Entry Date': i,
                'Entry Price': entry_price,
                'Position': position,
                'Quantity': qty
            })

        # Exit the Buy position if signal is 'Sell' or at session close
        if position == 'Buy' and (signal == 'Sell' or current_time == session_close):
            exit_price = ltp
            pnl = (exit_price - entry_price) * qty  # Calculate profit/loss
            txn_cost = (entry_price + exit_price) * qty * brokerage_rate  # Brokerage
            net_pnl = pnl - txn_cost  # Net profit/loss
            capital += net_pnl  # Update capital
            tradebook[-1].update({
                'Exit Date': i,
                'Exit Price': exit_price,
                'PnL': pnl,
                'Transaction Cost': txn_cost,
                'Net PnL': net_pnl,
                'Final Capital': capital
            })
            position = None  # Reset position

    # Convert tradebook to a DataFrame
    tradebook_df = pd.DataFrame(tradebook)

    # Final portfolio value
    final_portfolio_value = capital

    return tradebook_df, final_portfolio_value

# Run the backtest
tradebook, final_value = backtest_strategy(df)

# Display results
print("Tradebook:")
print(tradebook)
print(f"\nFinal Portfolio Value: ₹{final_value:,.2f}")

# Save the tradebook to CSV
tradebook.to_csv("tradebook.csv", index=False)
